In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
import lightgbm as lgb
from sklearn.metrics import classification_report, accuracy_score
from transformers import BertTokenizer, BertModel
import torch
from tqdm import tqdm

In [ ]:
train_df = pd.read_csv('/content/testing_data_binary.csv') #Path to binary train data
test_df = pd.read_csv('/content/testing_data_binary.csv') #Path to binary test data

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

In [ ]:
def extract_bert_features_with_progress(texts, tokenizer, model, max_length=128):
    embeddings = []
    for text in tqdm(texts, desc="Extracting BERT Features"):
        tokens = tokenizer.encode_plus(
            text,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        with torch.no_grad():  # Disable gradient
            output = model(
                input_ids=tokens['input_ids'],
                attention_mask=tokens['attention_mask']
            )
            # [CLS] token embedding
            embedding = output.last_hidden_state[:, 0, :].squeeze().numpy()
            embeddings.append(embedding)
    return np.array(embeddings)

In [ ]:
X_train = extract_bert_features_with_progress(train_df['prompt'], tokenizer, bert_model)
X_test = extract_bert_features_with_progress(test_df['prompt'], tokenizer, bert_model)
y_train = train_df['jailbreak']
y_test = test_df['jailbreak']

In [ ]:
logistic_model = LogisticRegression(max_iter=1000)
extra_trees_model = ExtraTreesClassifier(n_estimators=100, random_state=42)
lgbm_model = lgb.LGBMClassifier(random_state=42)

In [ ]:
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score, confusion_matrix
)
results = []

In [ ]:
def calculate_metrics(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    tpr = tp / (tp + fn)
    fnr = fn / (tp + fn)

    results.append({
        "Model": model_name,
        "Accuracy": round(accuracy, 4),
        "F1 Score": round(f1, 4),
        "TPR": round(tpr, 4),
        "FNR": round(fnr, 4)
    })

In [ ]:
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
calculate_metrics(y_test, logistic_predictions, "Logistic Regression")
extra_trees_model.fit(X_train, y_train)
extra_trees_predictions = extra_trees_model.predict(X_test)
calculate_metrics(y_test, extra_trees_predictions, "Extra Trees Classifier")
lgbm_model.fit(X_train, y_train)
lgbm_predictions = lgbm_model.predict(X_test)
calculate_metrics(y_test, lgbm_predictions, "LightGBM")

In [ ]:
results_df = pd.DataFrame(results)
results_df